In [ ]:
"""# WordyPy Bot Solution
This notebook contains a clean implementation of the `Letter` and `Bot` classes to solve the WordyPy assignment.
The Bot class makes smart guesses based on feedback.
"""

In [ ]:

class Letter:
    def __init__(self, letter: str, index: int) -> None:
        self.letter: str = letter.lower()
        self.index: int = index
        self.in_word: bool = False
        self.in_correct_place: bool = False

    def set_in_word(self, value: bool) -> None:
        self.in_word = value

    def set_in_correct_place(self, value: bool) -> None:
        self.in_correct_place = value

    def __repr__(self) -> str:
        return f"Letter({self.letter}, idx={self.index}, in_word={self.in_word}, correct={self.in_correct_place})"


In [ ]:

import random
from collections import Counter

class Bot:
    def __init__(self, word_file: str) -> None:
        with open(word_file, "r") as f:
            self.word_list = [w.strip().lower() for w in f if len(w.strip()) == 5]
        
        self.past_guesses = set()
        self.known_positions = {}   # index -> letter
        self.not_in_word = set()    # discarded letters

    def make_guess(self) -> str:
        # Filter candidates based on known constraints
        candidates = []
        for word in self.word_list:
            if word in self.past_guesses:
                continue

            if any(self.known_positions.get(i) and word[i] != self.known_positions[i]
                   for i in self.known_positions):
                continue

            if any(ch in self.not_in_word for ch in word):
                continue

            candidates.append(word)

        # Smart scoring: prioritize words with frequent letters
        word_pool = candidates if candidates else self.word_list
        all_letters = "".join(word_pool)
        freq = Counter(all_letters)

        def score(w):
            return sum(freq[ch] for ch in set(w))

        guess = max(word_pool, key=score)
        self.past_guesses.add(guess)
        return guess

    def record_guess_results(self, guess: str, feedback: list[Letter]) -> None:
        for letter in feedback:
            if letter.in_correct_place:
                self.known_positions[letter.index] = letter.letter
            elif not letter.in_word:
                self.not_in_word.add(letter.letter)
